# BEVPlace++ Kaggle Smoke-test Notebook

This notebook installs dependencies and runs a 1-epoch smoke-test of `train_distill.py` on Kaggle.

Before running: upload the following as Kaggle Datasets and add them to the kernel via **Add data**:
- **project-repo**: your repository files or a zip of the repo (contains `train_distill.py`, `REIN.py`, etc.)
- **teacher-model**: dataset containing `model_best.pth.tar` (path in kernel: `/kaggle/input/teacher-model/model_best.pth.tar`)
- (optional) **bev-dataset**: a small subset of BEV images for KITTI/NCLT to speed smoke-test.

Edit the dataset name variables in the next cell to match your uploaded dataset names.

In [ ]:
# List mounted datasets
!ls -la /kaggle/input || true

In [ ]:
# Set dataset names (edit these)
REPO_DATASET = 'project-repo'  # replace with your repo dataset name
TEACHER_DATASET = 'teacher-model'  # replace with your teacher model dataset name
BEV_DATASET = 'bev-dataset'  # optional small dataset name

In [ ]:
# Copy project files into working directory (adjust if your repo is a zip)
mkdir -p /kaggle/working/project
# If dataset is a folder of files, copy; if it's a zip, unzip accordingly
if [ -d "/kaggle/input/$REPO_DATASET" ]; then
  cp -r /kaggle/input/$REPO_DATASET/* /kaggle/working/project/
fi
if [ -f "/kaggle/input/$REPO_DATASET.zip" ]; then
  unzip -q /kaggle/input/$REPO_DATASET.zip -d /kaggle/working/project/
fi
ls -la /kaggle/working/project | sed -n '1,120p'

In [ ]:
# Install dependencies (CPU-friendly). Edit if you want GPU and Kaggle has CUDA available.
pip install --upgrade pip
pip install --index-url https://download.pytorch.org/whl/cpu torch torchvision torchaudio --quiet
pip install faiss-cpu --quiet || true
pip install -r /kaggle/working/project/requirements.txt --quiet || true

In [ ]:
# Sanity checks: python, torch, files
import sys, os
print('Python', sys.version)
try:
  import torch
  print('torch', torch.__version__, 'cuda available=', torch.cuda.is_available())
except Exception as e:
  print('torch import failed:', e)
print('Teacher model exists at:')
print(os.path.exists(f'/kaggle/input/{TEACHER_DATASET}/model_best.pth.tar'))

In [ ]:
# Run a 1-epoch smoke test (adjust arguments as needed).
# Ensure teacher_path points to the uploaded teacher dataset above.
%bash -x
cd /kaggle/working/project
python train_distill.py --batchSize 1 --nEpochs 1 --threads 2 --teacher_path /kaggle/input/$TEACHER_DATASET/model_best.pth.tar 2>&1 | tee smoke_test.log

## Troubleshooting & notes
- If you see `ModuleNotFoundError: No module named 'torch'`, re-run the install cell or enable Kaggle GPU runtime and install CUDA versions.
- If FAISS installation fails, try `pip uninstall faiss-cpu` and then `pip install faiss-cpu` or use conda in a custom environment (Kaggle CPUs often accept `faiss-cpu`).
- Output logs are stored at `/kaggle/working/project/smoke_test.log`. Model checkpoints (runs) will be under `/kaggle/working/project/runs_distill/` by default.
- Remember to set `REPO_DATASET` and `TEACHER_DATASET` variables above to your actual dataset names in the kernel.